# 14 — Prompt Evaluation

## Scenario
Northstar receives a variety of customer support tickets. We want to automatically route them to the correct department (`Refund`, `Technical Support`, or `Other`).

**The Problem:** How do we know if changing a prompt makes it "better"? Anecdotal testing ("vibe checks") doesn't scale. We need a deterministic, programmatic way to evaluate prompt performance over a frozen dataset.

In [ ]:
import os
from enum import Enum
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# 1. Define the Schema
class TicketCategory(str, Enum):
    REFUND = "Refund"
    TECHNICAL = "Technical Support"
    OTHER = "Other"

class TicketClassification(BaseModel):
    category: TicketCategory

# 2. Define the "Golden" Dataset
dataset = [
    {"ticket": "My mug arrived broken, I want my money back.", "expected": "Refund"},
    {"ticket": "The website keeps crashing when I try to log in.", "expected": "Technical Support"},
    {"ticket": "What time does the store open?", "expected": "Other"},
    # Edge case: Mentioning the word 'refund' but not actually asking for one
    {"ticket": "I know your refund policy is strict, but my mug's handle just snapped off after normal use. How do I fix it?", "expected": "Technical Support"},
]


## Step 1: The Baseline Evaluation

We write a naive prompt and run it against our dataset. We use Pydantic to ensure the model's output is strictly typed, making automated grading trivial.

In [ ]:
baseline_prompt_template = """\nClassify the following customer ticket.\n\nTicket: {ticket}\n"""

def evaluate_prompt(prompt_template, dataset):
    correct = 0
    for i, data in enumerate(dataset):
        ticket = data["ticket"]
        expected = data["expected"]
        
        prompt = prompt_template.format(ticket=ticket)
        
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=TicketClassification,
            )
        )
        
        # The model returns JSON matching the Pydantic schema
        classification = TicketClassification.model_validate_json(response.text)
        actual = classification.category.value
        
        is_match = (actual == expected)
        if is_match:
            correct += 1
            
        print(f"[{'PASS' if is_match else 'FAIL'}] Ticket {i+1}: Expected '{expected}', Got '{actual}'")
        
    accuracy = (correct / len(dataset)) * 100
    print(f"\nTotal Accuracy: {accuracy:.1f}%")
    return accuracy

print("--- BASELINE EVALUATION ---")
baseline_accuracy = evaluate_prompt(baseline_prompt_template, dataset)


## Step 2: The Candidate Evaluation

The baseline likely fails the edge case because it sees the word "refund" and jumps to conclusions. We update our prompt (Candidate) with better instructions and re-run the *exact same evaluation loop* to prove the regression works.

In [ ]:
candidate_prompt_template = """\nYou are an expert customer service router for Northstar.\nClassify the following customer ticket.\n\nCRITICAL RULE: Do not classify a ticket as a 'Refund' just because it contains the word refund. \nOnly classify it as a 'Refund' if the user is explicitly requesting their money back. \nIf they are asking for help fixing something, it is 'Technical Support'.\n\nTicket: {ticket}\n"""

print("\n--- CANDIDATE EVALUATION ---")
candidate_accuracy = evaluate_prompt(candidate_prompt_template, dataset)

print(f"\nImprovement: +{candidate_accuracy - baseline_accuracy}% accuracy")


## Conclusion

By defining a golden dataset and using Pydantic schemas to force deterministic outputs, we can confidently iterate on prompts and mathematically prove that our changes are actually improvements, rather than just "feeling" better on a few manual test cases.